In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [9]:
import numpy as np
import pandas as pd
import os
import warnings
import gc
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import xgboost as xgb
warnings.filterwarnings("ignore")
print("Imports OK ✓")

Imports OK ✓


In [10]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "kende23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "be7a74e194c6b7c0d666a4938cfee49142c7d603"
mlflow.set_tracking_uri("https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow")
dagshub.init(repo_owner="kende23", repo_name="ieee-cis-fraud-detection", mlflow=True)

experiment_name = "XGBoost_Magic"
if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(experiment_name)
mlflow.set_experiment(experiment_name)
print("MLflow connected ✓")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=50329404-1a67-4924-aa7d-b3f9e59fe7d4&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5a8d4ab289c09e1b613c804fcf419dc358a5ea3ca54c31f99f2655bc3e49cb0b




Accessing as kende23

Initialized MLflow to track repo "kende23/ieee-cis-fraud-detection"

Repository kende23/ieee-cis-fraud-detection initialized!

MLflow connected ✓


In [12]:
path = "/kaggle/input/competitions/ieee-fraud-detection/"

print("Loading...")
train_tr = pd.read_csv(path + "train_transaction.csv")
train_id = pd.read_csv(path + "train_identity.csv")
test_tr  = pd.read_csv(path + "test_transaction.csv")
test_id  = pd.read_csv(path + "test_identity.csv")

train_id.columns = [c.replace("-", "_") for c in train_id.columns]
test_id.columns  = [c.replace("-", "_") for c in test_id.columns]

train = train_tr.merge(train_id, on="TransactionID", how="left")
test  = test_tr.merge(test_id,   on="TransactionID", how="left")
print(f"train: {train.shape}, test: {test.shape}")

v_cols = [1,3,4,6,8,11,
          13,14,17,20,23,26,27,30,
          36,37,40,41,44,47,48,
          54,56,59,62,65,67,68,70,
          76,78,80,82,86,88,89,91,
          107,108,111,115,117,120,121,123,
          124,127,129,130,136,
          138,139,142,147,156,162,165,
          166,167,168,176,178,180,182,
          187,203,207,209,
          210,211,212,216,217,220,
          221,222,223,224,225,226,
          228,229,235,
          235,240,
          258,272,274,277,
          294,
          303,305,307,309,310,320]
v_cols = [f"V{v}" for v in v_cols if f"V{v}" in train.columns]
print(f"Selected {len(v_cols)} V columns")

base_cols = ["TransactionID","isFraud","TransactionDT","TransactionAmt",
             "ProductCD","card1","card2","card3","card4","card5","card6",
             "addr1","addr2","dist1","dist2","P_emaildomain","R_emaildomain",
             "C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12","C13","C14",
             "D1","D2","D3","D4","D5","D6","D7","D8","D9","D10","D11","D12","D13","D14","D15",
             "M1","M2","M3","M4","M5","M6","M7","M8","M9"]

train_cols = [c for c in base_cols if c in train.columns] + v_cols
test_cols  = [c for c in base_cols if c in test.columns and c != "isFraud"] + v_cols

train = train[[c for c in train_cols if c in train.columns]]
test  = test[[c for c in test_cols  if c in test.columns]]

print(f"After column selection: train={train.shape}, test={test.shape}")
print(f"Fraud rate: {train['isFraud'].mean():.4f}")

Loading...
train: (590540, 434), test: (506691, 433)
Selected 96 V columns
After column selection: train=(590540, 151), test=(506691, 150)
Fraud rate: 0.0350


In [14]:
print("=== FEATURE ENGINEERING (cdeotte approach) ===")

# reload fresh to avoid duplicate columns from previous runs
path = "/kaggle/input/competitions/ieee-fraud-detection/"
train_tr = pd.read_csv(path + "train_transaction.csv")
train_id = pd.read_csv(path + "train_identity.csv")
test_tr  = pd.read_csv(path + "test_transaction.csv")
test_id  = pd.read_csv(path + "test_identity.csv")

train_id.columns = [c.replace("-", "_") for c in train_id.columns]
test_id.columns  = [c.replace("-", "_") for c in test_id.columns]

train = train_tr.merge(train_id, on="TransactionID", how="left")
test  = test_tr.merge(test_id,   on="TransactionID", how="left")

# select specific V columns (cdeotte's selection)
v_cols = [1,3,4,6,8,11,
          13,14,17,20,23,26,27,30,
          36,37,40,41,44,47,48,
          54,56,59,62,65,67,68,70,
          76,78,80,82,86,88,89,91,
          107,108,111,115,117,120,121,123,
          124,127,129,130,136,
          138,139,142,147,156,162,165,
          166,167,168,176,178,180,182,
          187,203,207,209,
          210,211,212,216,217,220,
          221,222,223,224,225,226,
          228,229,235,240,
          258,272,274,277,
          294,303,305,307,309,310,320]
v_cols = [f"V{v}" for v in v_cols if f"V{v}" in train.columns]
print(f"Selected {len(v_cols)} V columns")

base_cols = ["TransactionID","isFraud","TransactionDT","TransactionAmt",
             "ProductCD","card1","card2","card3","card4","card5","card6",
             "addr1","addr2","dist1","dist2","P_emaildomain","R_emaildomain",
             "C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12","C13","C14",
             "D1","D2","D3","D4","D5","D6","D7","D8","D9","D10","D11","D12","D13","D14","D15",
             "M1","M2","M3","M4","M5","M6","M7","M8","M9"]

train_cols = [c for c in base_cols if c in train.columns] + v_cols
test_cols  = [c for c in base_cols if c in test.columns and c != "isFraud"] + v_cols

train = train[[c for c in train_cols if c in train.columns]]
test  = test[[c for c in test_cols  if c in test.columns]]

# verify no duplicate columns
assert train.columns.duplicated().sum() == 0, "Duplicate cols in train!"
assert test.columns.duplicated().sum() == 0,  "Duplicate cols in test!"
print(f"train: {train.shape}, test: {test.shape}")
print(f"Fraud rate: {train['isFraud'].mean():.4f}")

# now combine
print("\nCombining train+test for aggregations...")
train["is_train"] = 1
test["is_train"]  = 0
combined = pd.concat([train, test], ignore_index=True, sort=False)
print(f"Combined: {combined.shape}")

# ── time features ──
print("Adding time features...")
combined["TransactionDay"]  = (combined["TransactionDT"] / 86400).astype(np.float32)
combined["TransactionHour"] = ((combined["TransactionDT"] / 3600) % 24).astype(np.float32)

# ── D normalization ──
print("Normalizing D columns...")
for d in [1,2,3,4,10,15]:
    if f"D{d}" in combined.columns:
        combined[f"D{d}n"] = combined["TransactionDay"] - combined[f"D{d}"]

# ── cents ──
combined["cents"] = (combined["TransactionAmt"] - np.floor(combined["TransactionAmt"])).astype(np.float32)

# ── email provider ──
combined["P_email_provider"] = combined["P_emaildomain"].str.split(".").str[0].fillna("unknown")
combined["R_email_provider"] = combined["R_emaildomain"].str.split(".").str[0].fillna("unknown")

# ── frequency encoding ──
print("Frequency encoding...")
for col in ["addr1", "card1", "card2", "card3", "P_emaildomain"]:
    freq = combined[col].value_counts(dropna=False)
    combined[f"{col}_FE"] = combined[col].map(freq).astype(np.float32)
    print(f"  {col}_FE done")

# ── UID ──
print("Creating UIDs...")
combined["uid"] = (
    combined["card1"].astype(str) + "_" +
    combined["addr1"].astype(str) + "_" +
    combined["P_emaildomain"].astype(str)
)
combined["uid2"] = (
    combined["card1"].astype(str) + "_" +
    combined["addr1"].astype(str)
)
print(f"  Unique UIDs:  {combined['uid'].nunique()}")
print(f"  Unique UID2s: {combined['uid2'].nunique()}")

uid_freq  = combined["uid"].value_counts(dropna=False)
uid2_freq = combined["uid2"].value_counts(dropna=False)
combined["uid_FE"]  = combined["uid"].map(uid_freq).astype(np.float32)
combined["uid2_FE"] = combined["uid2"].map(uid2_freq).astype(np.float32)

# ── aggregations ──
print("Aggregations...")
for grp in ["card1", "uid2"]:
    for feat in ["TransactionAmt", "D9", "D11"]:
        if feat in combined.columns:
            combined[f"{feat}_{grp}_mean"] = combined[grp].map(
                combined.groupby(grp)[feat].mean()).astype(np.float32)
            combined[f"{feat}_{grp}_std"]  = combined[grp].map(
                combined.groupby(grp)[feat].std()).astype(np.float32)
            print(f"  {feat}_{grp} done")

# ── drop uid strings ──
combined = combined.drop(columns=["uid", "uid2"])

# ── split back ──
print("\nSplitting back...")
train = combined[combined["is_train"] == 1].drop(columns=["is_train"])
test  = combined[combined["is_train"] == 0].drop(columns=["is_train", "isFraud"])
print(f"train: {train.shape}, test: {test.shape}")

del combined
gc.collect()

X = train.drop(columns=["isFraud", "TransactionID"])
y = train["isFraud"]
test = test.drop(columns=["TransactionID"])

common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
assert list(X.columns) == list(test.columns)
cols_after_engineering = X.shape[1]
print(f"\nFinal: X={X.shape}, test={test.shape}")
print("Feature engineering done ✓")

=== FEATURE ENGINEERING (cdeotte approach) ===
Selected 95 V columns
train: (590540, 150), test: (506691, 149)
Fraud rate: 0.0350

Combining train+test for aggregations...
Combined: (1097231, 151)
Adding time features...
Normalizing D columns...
Frequency encoding...
  addr1_FE done
  card1_FE done
  card2_FE done
  card3_FE done
  P_emaildomain_FE done
Creating UIDs...
  Unique UIDs:  130520
  Unique UID2s: 55370
Aggregations...
  TransactionAmt_card1 done
  D9_card1 done
  D11_card1 done
  TransactionAmt_uid2 done
  D9_uid2 done
  D11_uid2 done

Splitting back...
train: (590540, 180), test: (506691, 179)

Final: X=(590540, 178), test=(506691, 178)
Feature engineering done ✓


In [15]:
print("=== FEATURE SELECTION ===")
print(f"Starting: X={X.shape}")

# drop high missing >90%
missing_rate = X.isnull().mean()
high_missing = missing_rate[missing_rate > 0.9].index.tolist()
print(f"Dropping {len(high_missing)} high missing cols")
X    = X.drop(columns=high_missing)
test = test.drop(columns=high_missing)

# drop near-zero variance
num_cols_temp = X.select_dtypes(include=[np.number]).columns
low_var = [c for c in num_cols_temp if X[c].std() < 0.01]
print(f"Dropping {len(low_var)} low variance cols: {low_var}")
X    = X.drop(columns=low_var)
test = test.drop(columns=low_var)

# drop high cardinality categoricals only
cat_cols_temp = X.select_dtypes(include=["object"]).columns
high_card = [c for c in cat_cols_temp if X[c].nunique() > 200]
print(f"Dropping {len(high_card)} high cardinality cols: {high_card}")
X    = X.drop(columns=high_card)
test = test.drop(columns=high_card)

# NOTE: no aggressive correlation dropping this time
# cdeotte kept ~250 features, we trust XGBoost to handle correlations

assert list(X.columns) == list(test.columns)
cols_after_cleaning = X.shape[1]
final_feature_count = X.shape[1]
final_cols = X.columns.tolist()
print(f"\nFinal: X={X.shape}")
print("✓ No aggressive correlation dropping — XGBoost handles this internally")

=== FEATURE SELECTION ===
Starting: X=(590540, 178)
Dropping 2 high missing cols
Dropping 2 low variance cols: ['V1', 'V305']
Dropping 0 high cardinality cols: []

Final: X=(590540, 174)
✓ No aggressive correlation dropping — XGBoost handles this internally


In [ ]:
print("=== MEMORY REDUCTION ===")
def reduce_mem(df, name):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    print(f"  {name}: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
    return df

X    = reduce_mem(X,    "X")
test = reduce_mem(test, "test")
print(X.dtypes.value_counts())

In [16]:
print("=== TIME-BASED SPLIT ===")
X = X.sort_values("TransactionDT")
y = y[X.index].reset_index(drop=True)
X = X.reset_index(drop=True)

split_point = X["TransactionDT"].quantile(0.8)
print(f"Split at 80th percentile: {split_point:.0f}")

train_mask = X["TransactionDT"] <= split_point
val_mask   = X["TransactionDT"] >  split_point

X_train = X[train_mask].reset_index(drop=True)
X_val   = X[val_mask].reset_index(drop=True)
y_train = y[train_mask].reset_index(drop=True)
y_val   = y[val_mask].reset_index(drop=True)

print(f"X_train: {X_train.shape}, fraud: {y_train.mean():.4f}")
print(f"X_val:   {X_val.shape},   fraud: {y_val.mean():.4f}")

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")
print(f"Cat cols: {cat_cols}")

del X
gc.collect()
print("Freed X ✓")

=== TIME-BASED SPLIT ===
Split at 80th percentile: 12192854
X_train: (472432, 174), fraud: 0.0351
X_val:   (118108, 174),   fraud: 0.0344
Numeric: 158, Categorical: 16
Cat cols: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'P_email_provider', 'R_email_provider']
Freed X ✓


In [17]:
from sklearn.model_selection import RandomizedSearchCV

print("=== XGBOOST MAGIC TRAINING ===")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

fraud_count      = int(y_train.sum())
nonfraud_count   = int((y_train == 0).sum())
scale_pos_weight = nonfraud_count / fraud_count
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

base_xgb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="auc",
        tree_method="hist",
        device="cuda" if os.path.exists("/usr/local/cuda") else "cpu",
        n_jobs=-1
    ))
])

param_dist = {
    "classifier__n_estimators":     [300, 500, 700],
    "classifier__max_depth":        [4, 5, 6],
    "classifier__learning_rate":    [0.01, 0.02, 0.05],
    "classifier__subsample":        [0.7, 0.8, 0.9],
    "classifier__colsample_bytree": [0.5, 0.6, 0.7],
    "classifier__min_child_weight": [50, 100, 200],
    "classifier__reg_alpha":        [0.1, 0.5, 1.0],
    "classifier__reg_lambda":       [0.5, 1.0, 2.0],
}

print("Running RandomizedSearchCV (10 trials, 3-fold)...")
search = RandomizedSearchCV(
    base_xgb,
    param_distributions=param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    verbose=2,
    n_jobs=1
)
search.fit(X_train, y_train)

print(f"\nBest CV AUC: {search.best_score_:.4f}")
model_xgb = search.best_estimator_

# evaluate
val_probs = model_xgb.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)
val_auc       = roc_auc_score(y_val, val_probs)
val_precision = precision_score(y_val, val_preds)
val_recall    = recall_score(y_val, val_preds)
val_f1        = f1_score(y_val, val_preds)

print(f"\n=== VALIDATION METRICS ===")
print(f"  AUC:       {val_auc:.4f}")
print(f"  Precision: {val_precision:.4f}")
print(f"  Recall:    {val_recall:.4f}")
print(f"  F1:        {val_f1:.4f}")

# show all trials
print("\n=== ALL TRIALS ===")
results_df = pd.DataFrame(search.cv_results_).sort_values("mean_test_score", ascending=False)
for _, row in results_df.iterrows():
    p = {k.replace("classifier__", ""): v for k, v in row["params"].items()}
    print(f"  Rank {int(row['rank_test_score'])}: AUC={row['mean_test_score']:.4f} ±{row['std_test_score']:.4f}")

# log cleaning
with mlflow.start_run(run_name="XGBoost_Magic_Cleaning"):
    mlflow.log_param("missing_threshold",    0.9)
    mlflow.log_param("variance_threshold",   0.01)
    mlflow.log_param("high_card_threshold",  200)
    mlflow.log_param("corr_drop",           "NONE - XGBoost handles internally")
    mlflow.log_param("dropped_high_missing", len(high_missing))
    mlflow.log_param("dropped_low_variance", len(low_var))
    mlflow.log_param("dropped_high_card",    len(high_card))
    mlflow.log_param("cols_after_cleaning",  cols_after_cleaning)
    print(f"  Cleaning Run ID: {mlflow.active_run().info.run_id}")

# log feature engineering
with mlflow.start_run(run_name="XGBoost_Magic_Feature_Engineering"):
    mlflow.log_param("cols_before", 432)
    mlflow.log_param("cols_after",  cols_after_engineering)
    mlflow.log_param("v_cols_selected",      len(v_cols))
    mlflow.log_param("uid_card1_addr1_email", True)
    mlflow.log_param("frequency_encoding",   True)
    mlflow.log_param("D_normalization",      True)
    mlflow.log_param("TransactionAmt_agg",   True)
    mlflow.log_param("D9_D11_agg",           True)
    mlflow.log_param("combined_train_test_for_agg", True)
    mlflow.log_param("validation_strategy",  "time_based_80_20")
    print(f"  FE Run ID: {mlflow.active_run().info.run_id}")

# log feature selection
with mlflow.start_run(run_name="XGBoost_Magic_Feature_Selection"):
    mlflow.log_param("final_feature_count", final_feature_count)
    with open("magic_final_features.txt", "w") as f:
        f.write("\n".join(final_cols))
    mlflow.log_artifact("magic_final_features.txt")
    print(f"  FS Run ID: {mlflow.active_run().info.run_id}")

# log each trial
results_df2 = pd.DataFrame(search.cv_results_)
for i, row in results_df2.iterrows():
    trial_params = {k.replace("classifier__", ""): v for k, v in row["params"].items()}
    trial_params["scale_pos_weight"] = round(scale_pos_weight, 2)
    with mlflow.start_run(run_name=f"XGBoost_Magic_Trial_{i+1}"):
        mlflow.log_params(trial_params)
        mlflow.log_metrics({
            "cv_auc_mean": round(row["mean_test_score"], 4),
            "cv_auc_std":  round(row["std_test_score"],  4),
            "cv_rank":     int(row["rank_test_score"]),
        })
        print(f"  Trial {i+1} — AUC: {row['mean_test_score']:.4f}")

# log best model
best_params = {k.replace("classifier__", ""): v for k, v in search.best_params_.items()}
best_params["scale_pos_weight"]    = round(scale_pos_weight, 2)
best_params["validation_strategy"] = "time_based_80_20"
best_params["feature_approach"]    = "cdeotte_magic"

with mlflow.start_run(run_name="XGBoost_Magic_Best"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({
        "val_auc":       val_auc,
        "val_precision": val_precision,
        "val_recall":    val_recall,
        "val_f1":        val_f1,
        "cv_auc":        round(search.best_score_, 4),
    })
    mlflow.sklearn.log_model(
        model_xgb,
        artifact_path="xgboost_magic_pipeline",
        registered_model_name="XGBoost_Magic_Pipeline"
    )
    print(f"  Best Run ID: {mlflow.active_run().info.run_id}")

print("\nAll logged ✓")
del X_train
del X_val
gc.collect()
print("Memory freed ✓")

=== XGBOOST MAGIC TRAINING ===
scale_pos_weight: 27.46
Running RandomizedSearchCV (10 trials, 3-fold)...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END classifier__colsample_bytree=0.5, classifier__learning_rate=0.02, classifier__max_depth=4, classifier__min_child_weight=100, classifier__n_estimators=500, classifier__reg_alpha=1.0, classifier__reg_lambda=1.0, classifier__subsample=0.9; total time=  38.8s
[CV] END classifier__colsample_bytree=0.5, classifier__learning_rate=0.02, classifier__max_depth=4, classifier__min_child_weight=100, classifier__n_estimators=500, classifier__reg_alpha=1.0, classifier__reg_lambda=1.0, classifier__subsample=0.9; total time=  38.6s
[CV] END classifier__colsample_bytree=0.5, classifier__learning_rate=0.02, classifier__max_depth=4, classifier__min_child_weight=100, classifier__n_estimators=500, classifier__reg_alpha=1.0, classifier__reg_lambda=1.0, classifier__subsample=0.9; total time=  37.9s
[CV] END classifier__colsample_bytree=0

2026/05/03 16:07:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 16:07:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGBoost_Magic_Pipeline'.
2026/05/03 16:08:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_Magic_Pipeline, version 1
Created version '1' of model 'XGBoost_Magic_Pipeline'.


  Best Run ID: 6537eff0e5074cfb8f8491957e1db74b
🏃 View run XGBoost_Magic_Best at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/6537eff0e5074cfb8f8491957e1db74b
🧪 View experiment at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/4

All logged ✓
Memory freed ✓


In [18]:
print("=== GENERATING SUBMISSION ===")

test_probs = model_xgb.predict_proba(test)[:, 1]
print(f"  Raw predictions: {test_probs.shape}")
print(f"  Raw fraud rate:  {test_probs.mean():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_tr["TransactionID"].values,
    "isFraud": test_probs
})

# ── post processing (cdeotte trick — +0.001 LB) ──
# replace each client's predictions with their average prediction
print("\nApplying post-processing (UID averaging)...")
test_tr["uid"] = (
    test_tr["card1"].astype(str) + "_" +
    test_tr["addr1"].astype(str) + "_" +
    test_tr["P_emaildomain"].astype(str)
)
submission["uid"] = test_tr["uid"].values
uid_mean = submission.groupby("uid")["isFraud"].mean()
submission["isFraud_pp"] = submission["uid"].map(uid_mean)
print(f"  Before PP fraud rate: {submission['isFraud'].mean():.4f}")
print(f"  After PP fraud rate:  {submission['isFraud_pp'].mean():.4f}")

# save both versions
submission[["TransactionID","isFraud"]].to_csv("submission_magic_raw.csv", index=False)
submission[["TransactionID"]].assign(isFraud=submission["isFraud_pp"]).to_csv("submission_magic_pp.csv", index=False)

print(f"\nSubmission shape: {submission.shape}")
print(submission[["TransactionID","isFraud","isFraud_pp"]].head(10))
print("\nsubmission_magic_raw.csv — without post-processing")
print("submission_magic_pp.csv  — with UID averaging post-processing")
print("\nSubmit BOTH and compare scores!")
print("Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions")

=== GENERATING SUBMISSION ===
  Raw predictions: (506691,)
  Raw fraud rate:  0.2552

Applying post-processing (UID averaging)...
  Before PP fraud rate: 0.2552
  After PP fraud rate:  0.2552

Submission shape: (506691, 4)
   TransactionID   isFraud  isFraud_pp
0        3663549  0.055095    0.144028
1        3663550  0.231558    0.178056
2        3663551  0.112466    0.115213
3        3663552  0.066912    0.243049
4        3663553  0.215912    0.183214
5        3663554  0.114546    0.215143
6        3663555  0.371952    0.141320
7        3663556  0.511645    0.292358
8        3663557  0.015570    0.069828
9        3663558  0.341003    0.275598

submission_magic_raw.csv — without post-processing
submission_magic_pp.csv  — with UID averaging post-processing

Submit BOTH and compare scores!
Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions
